In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define path to the pre-processed SMARD dataset
file_path = (
    "/content/drive/MyDrive/Colab Notebooks/SMARD_Cleaned_20260806_20260816.csv"
)

# Verify file existence
if os.path.exists(file_path):
  print("Dataset found successfully! Proceeding with data loading...")
else:
  print(
      "File not found! Please verify the folder structure in your Google Drive."
  )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset found successfully! Proceeding with data loading...


In [3]:
# Cell 1: Imports and Environment Setup
# در صورت نیاز: pip install pulp plotly pandas numpy

import numpy as np
import pandas as pd
import pulp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print(f"PuLP Version: {pulp.__version__}")
print("محیط بهینه‌سازی و تحلیل بازار آماده است.")

PuLP Version: 3.3.2
محیط بهینه‌سازی و تحلیل بازار آماده است.


In [4]:
# Cell 2: Market Data Generation (EPEX Spot DA & German aFRR Capacity Prices)

# تعریف ۲۴ ساعت شبانه‌روز
hours = list(range(24))

# ۱. قیمت بازار اسپات روزانه آلمان (EPEX Spot Day-Ahead) - بر حسب €/MWh
# الگوی مشخص آلمان: قیمت ارزان یا منفی در ظهر (پیک خورشیدی) و قیمت‌های بالا در پیک صبح و عصر
spot_prices = [
    58.2, 52.1, 48.5, 45.0, 51.4, 72.8,        # 00:00 - 05:00 (شبانه)
    125.6, 168.4, 142.0, 95.5, 62.0, 31.2,     # 06:00 - 11:00 (پیک صبحگاهی و افت خورشیدی)
    18.5, 12.0, 24.8, 55.3, 110.0, 185.5,      # 12:00 - 17:00 (پایین‌ترین قیمت خورشیدی و شروع پیک عصر)
    215.0, 192.4, 145.0, 112.5, 88.0, 68.2     # 18:00 - 23:00 (پیک عصرگاهی و افت شبانه)
]

# ۲. قیمت‌های ظرفیت aFRR در پلتفرم Regelleistung.net آلمان (بر حسب €/MW/h)
# در آلمان حراج ظرفیت در ۶ بلوک ۴ ساعته برگزار می‌شود:
# بلوک ۱: 00-04 | بلوک ۲: 04-08 | بلوک ۳: 08-12 | بلوک ۴: 12-16 | بلوک ۵: 16-20 | بلوک ۶: 20-24
afrr_pos_block_prices = [18.5, 38.0, 26.5, 22.0, 48.0, 32.5] # رزرو مثبت (+aFRR)
afrr_neg_block_prices = [32.0, 16.5, 24.0, 42.5, 15.0, 22.0] # رزرو منفی (-aFRR: ظهر به دلیل وفور PV بالاست)

# نگاشت بلوک‌های ۴ ساعته به ۲۴ ساعت
afrr_pos_prices = []
afrr_neg_prices = []
for h in hours:
    block_idx = h // 4
    afrr_pos_prices.append(afrr_pos_block_prices[block_idx])
    afrr_neg_prices.append(afrr_neg_block_prices[block_idx])

df_market = pd.DataFrame({
    'Hour': hours,
    'Block_4h': [h // 4 for h in hours],
    'Spot_DA_EUR_MWh': spot_prices,
    'aFRR_Pos_Cap_EUR_MW': afrr_pos_prices,
    'aFRR_Neg_Cap_EUR_MW': afrr_neg_prices
})

df_market.head(8)

,Hour,Block_4h,Spot_DA_EUR_MWh,aFRR_Pos_Cap_EUR_MW,aFRR_Neg_Cap_EUR_MW
0,0,0,58.2,18.5,32.0
1,1,0,52.1,18.5,32.0
2,2,0,48.5,18.5,32.0
3,3,0,45.0,18.5,32.0
4,4,1,51.4,38.0,16.5
5,5,1,72.8,38.0,16.5
6,6,1,125.6,38.0,16.5
7,7,1,168.4,38.0,16.5


In [5]:
# Cell 3: Technical Specifications of the Battery Asset (10 MW / 20 MWh System)

BESS_CONFIG = {
    'P_max': 10.0,            # حداکثر توان نامی اینورتر (MW)
    'E_max': 20.0,            # ظرفیت اسمی ذخیره انرژی (MWh)
    'SOC_min_pct': 0.10,      # حداقل سطح شارژ مجاز (۱۰ درصد)
    'SOC_max_pct': 0.90,      # حداکثر سطح شارژ مجاز (۹۰ درصد)
    'SOC_init_pct': 0.50,     # سطح شارژ اولیه باتری (۵۰ درصد)
    'eta_ch': 0.95,           # راندمان شارژ
    'eta_dis': 0.95,          # راندمان دشارژ (RTE کل ~ 90.25%)
    'deg_cost': 5.0,          # هزینه استهلاک/تخریب سلول به ازای هر MWh تبادل توان (€/MWh)
    'afrr_dur_buffer': 0.5    # بافر زمانی تعهد aFRR برای حفاظت از SOC (معادل 30 دقیقه تحویل پیوسته)
}

# مقادیر بر حسب MWh
SOC_MIN = BESS_CONFIG['SOC_min_pct'] * BESS_CONFIG['E_max']
SOC_MAX = BESS_CONFIG['SOC_max_pct'] * BESS_CONFIG['E_max']
SOC_INIT = BESS_CONFIG['SOC_init_pct'] * BESS_CONFIG['E_max']

In [6]:
# Cell 4: MILP Co-Optimization Model Formulation in PuLP

def solve_co_optimization(df_market, config):
    model = pulp.LpProblem("BESS_German_DayAhead_aFRR_CoOptimization", pulp.LpMaximize)

    T = df_market['Hour'].tolist()
    num_blocks = 6
    blocks = list(range(num_blocks))

    # --- متغیرهای تصمیم (Decision Variables) ---
    # بازار اسپات روزانه
    p_ch = pulp.LpVariable.dicts("P_ch", T, lowBound=0, upBound=config['P_max'], cat=pulp.LpContinuous)
    p_dis = pulp.LpVariable.dicts("P_dis", T, lowBound=0, upBound=config['P_max'], cat=pulp.LpContinuous)
    u_ch = pulp.LpVariable.dicts("u_ch", T, cat=pulp.LpBinary)
    u_dis = pulp.LpVariable.dicts("u_dis", T, cat=pulp.LpBinary)

    # بازار aFRR: ظرفیت باید برای هر بلوک ۴ ساعته یکسان باشد (German Market Rule)
    r_pos_block = pulp.LpVariable.dicts("R_pos_block", blocks, lowBound=0, upBound=config['P_max'], cat=pulp.LpContinuous)
    r_neg_block = pulp.LpVariable.dicts("R_neg_block", blocks, lowBound=0, upBound=config['P_max'], cat=pulp.LpContinuous)

    # وضعیت شارژ باتری (SOC) در پایان هر ساعت
    soc = pulp.LpVariable.dicts("SOC", T, lowBound=SOC_MIN, upBound=SOC_MAX, cat=pulp.LpContinuous)

    # --- تابع هدف (Objective Function) ---
    # حداکثرسازی سود = درآمد اسپات + درآمد ظرفیت aFRR - هزینه استهلاک باتری
    spot_rev = pulp.lpSum([
        (df_market.loc[t, 'Spot_DA_EUR_MWh'] * p_dis[t] - df_market.loc[t, 'Spot_DA_EUR_MWh'] * p_ch[t])
        for t in T
    ])

    afrr_rev = pulp.lpSum([
        4 * (df_market.loc[b*4, 'aFRR_Pos_Cap_EUR_MW'] * r_pos_block[b] +
             df_market.loc[b*4, 'aFRR_Neg_Cap_EUR_MW'] * r_neg_block[b])
        for b in blocks
    ])

    deg_cost = pulp.lpSum([
        config['deg_cost'] * (p_ch[t] + p_dis[t])
        for t in T
    ])

    model += spot_rev + afrr_rev - deg_cost, "Total_Daily_Profit"

    # --- قیود سیستم (Constraints) ---
    for t in T:
        b = t // 4

        # ۱. جلوگیری از شارژ و دشارژ هم‌زمان در بازار اسپات
        model += p_ch[t] <= config['P_max'] * u_ch[t], f"Charge_Binary_{t}"
        model += p_dis[t] <= config['P_max'] * u_dis[t], f"Discharge_Binary_{t}"
        model += u_ch[t] + u_dis[t] <= 1, f"Exclusive_Mode_{t}"

        # ۲. محدودیت هدایت توان اینورتر (اشتراک ظرفیت بین اسپات و رزرو aFRR)
        # حداکثر توان تزریق به شبکه: توان دشارژ اسپات + توان آماده رزرو مثبت
        model += p_dis[t] + r_pos_block[b] <= config['P_max'], f"Max_Injection_Limit_{t}"
        # حداکثر توان دریافت از شبکه: توان شارژ اسپات + توان آماده رزرو منفی
        model += p_ch[t] + r_neg_block[b] <= config['P_max'], f"Max_Withdrawal_Limit_{t}"

        # ۳. دینامیک وضعیت شارژ باتری (SOC Tracking)
        prev_soc = SOC_INIT if t == 0 else soc[t - 1]
        model += soc[t] == prev_soc + (p_ch[t] * config['eta_ch'] - (p_dis[t] / config['eta_dis'])), f"SOC_Dynamic_{t}"

        # ۴. تضمین هدایت انرژی برای تحویل تعهد aFRR (Energy Buffer Constraint)
        # برای ارائه رزرو مثبت، باید انرژی کافی در باتری بماند
        model += soc[t] - (r_pos_block[b] * config['afrr_dur_buffer']) >= SOC_MIN, f"aFRR_Pos_Energy_Buffer_{t}"
        # برای ارائه رزرو منفی، باید فضای خالی برای جذب انرژی وجود داشته باشد
        model += soc[t] + (r_neg_block[b] * config['afrr_dur_buffer']) <= SOC_MAX, f"aFRR_Neg_Energy_Buffer_{t}"

    # ۵. قود انتهای روز: باتری باید در پایان ۲۴ ساعت حداقل به سطح شارژ اولیه برگردد
    model += soc[23] >= SOC_INIT, "Final_SOC_Neutrality"

    # حل مدل
    solver = pulp.PULP_CBC_CMD(msg=False)
    status = model.solve(solver)

    print(f"وضعیت بهینه‌سازی: {pulp.LpStatus[status]}")
    print(f"سود خالص کل محاسبه‌شده: {pulp.value(model.objective):,.2f} یورو")

    # استخراج نتایج در قالب دیتافریم
    results = []
    for t in T:
        b = t // 4
        results.append({
            'Hour': t,
            'Block_4h': b,
            'Spot_DA_EUR_MWh': df_market.loc[t, 'Spot_DA_EUR_MWh'],
            'aFRR_Pos_EUR_MW': df_market.loc[t, 'aFRR_Pos_Cap_EUR_MW'],
            'aFRR_Neg_EUR_MW': df_market.loc[t, 'aFRR_Neg_Cap_EUR_MW'],
            'P_charge_MW': p_ch[t].varValue,
            'P_discharge_MW': p_dis[t].varValue,
            'R_aFRR_Pos_MW': r_pos_block[b].varValue,
            'R_aFRR_Neg_MW': r_neg_block[b].varValue,
            'SOC_MWh': soc[t].varValue,
            'SOC_Pct': (soc[t].varValue / config['E_max']) * 100
        })

    return pd.DataFrame(results), model

df_results, opt_model = solve_co_optimization(df_market, BESS_CONFIG)

وضعیت بهینه‌سازی: Optimal
سود خالص کل محاسبه‌شده: 13,638.67 یورو


In [7]:
# Cell 5: Revenue Stacking Breakdown & Financial Summary

# تفکیک دقیق درآمدها
spot_arbitrage_rev = sum(
    df_results['Spot_DA_EUR_MWh'] * df_results['P_discharge_MW'] -
    df_results['Spot_DA_EUR_MWh'] * df_results['P_charge_MW']
)

afrr_pos_rev = sum(df_results['aFRR_Pos_EUR_MW'] * df_results['R_aFRR_Pos_MW'])
afrr_neg_rev = sum(df_results['aFRR_Neg_EUR_MW'] * df_results['R_aFRR_Neg_MW'])
total_afrr_rev = afrr_pos_rev + afrr_neg_rev

degradation_cost = sum(
    BESS_CONFIG['deg_cost'] * (df_results['P_charge_MW'] + df_results['P_discharge_MW'])
)

net_daily_profit = spot_arbitrage_rev + total_afrr_rev - degradation_cost

print("="*55)
print("📊 گزارش عملکرد پشته درآمدی باتری ۱۰ مگاوات / ۲۰ مگاوات‌ساعت")
print("="*55)
print(f"💰 درآمد آربیتراژ بازار اسپات (Day-Ahead):  {spot_arbitrage_rev:>10,.2f} €")
print(f"📈 درآمد رزرو مثبت (+aFRR Capacity):       {afrr_pos_rev:>10,.2f} €")
print(f"📉 درآمد رزرو منفی (-aFRR Capacity):       {afrr_neg_rev:>10,.2f} €")
print(f"⚡ مجموع درآمد رزرو ثانویه (aFRR Total):   {total_afrr_rev:>10,.2f} €")
print(f"🔧 هزینه استهلاک چرخه شارژ/دشارژ:         -{degradation_cost:>10,.2f} €")
print("-"*55)
print(f"🏆 سود خالص روزانه (Net Daily Profit):     {net_daily_profit:>10,.2f} €")
print("="*55)

if net_daily_profit >= 6000:
    print(f"✅ هدف پروژه محقق شد: سود روزانه {net_daily_profit:,.2f} € بیش از ۶۰۰۰ یورو است!")
else:
    print(f"⚠️ سود روزانه به سقف ۶۰۰۰ یورو نرسیده است.")

📊 گزارش عملکرد پشته درآمدی باتری ۱۰ مگاوات / ۲۰ مگاوات‌ساعت
💰 درآمد آربیتراژ بازار اسپات (Day-Ahead):      510.21 €
📈 درآمد رزرو مثبت (+aFRR Capacity):         7,237.60 €
📉 درآمد رزرو منفی (-aFRR Capacity):         5,925.45 €
⚡ مجموع درآمد رزرو ثانویه (aFRR Total):    13,163.05 €
🔧 هزینه استهلاک چرخه شارژ/دشارژ:         -     34.59 €
-------------------------------------------------------
🏆 سود خالص روزانه (Net Daily Profit):      13,638.67 €
✅ هدف پروژه محقق شد: سود روزانه 13,638.67 € بیش از ۶۰۰۰ یورو است!


In [11]:
# Cell 6: Multi-Axis Interactive Visualization using Plotly (Perfect Legend & Layout Fix)

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=(
        "<b>1. German Market Price Signals (Spot vs. aFRR Capacity)</b>",
        "<b>2. BESS Power Dispatch & Reserve Allocation</b>",
        "<b>3. Battery State of Charge (SOC) & Cumulative Revenue Stacking</b>"
    ),
    specs=[[{"secondary_y": True}], [{"secondary_y": False}], [{"secondary_y": True}]]
)

time_labels = [f"{h:02d}:00" for h in df_results['Hour']]

# Chart 1: Market Prices
fig.add_trace(
    go.Scatter(x=time_labels, y=df_results['Spot_DA_EUR_MWh'], name="Spot Price (€/MWh)",
               line=dict(color='#2b5c8f', width=2.5)),
    row=1, col=1, secondary_y=False
)
fig.add_trace(
    go.Scatter(x=time_labels, y=df_results['aFRR_Pos_EUR_MW'], name="+aFRR Capacity Price (€/MW)",
               line=dict(color='#2ca02c', dash='dot', width=2)),
    row=1, col=1, secondary_y=True
)
fig.add_trace(
    go.Scatter(x=time_labels, y=df_results['aFRR_Neg_EUR_MW'], name="-aFRR Capacity Price (€/MW)",
               line=dict(color='#d62728', dash='dot', width=2)),
    row=1, col=1, secondary_y=True
)

# Chart 2: Power Dispatch & Reserve Commitments
fig.add_trace(
    go.Bar(x=time_labels, y=df_results['P_discharge_MW'], name="Spot Discharge (MW)",
           marker_color='#1f77b4'),
    row=2, col=1
)
fig.add_trace(
    go.Bar(x=time_labels, y=-df_results['P_charge_MW'], name="Spot Charge (MW)",
           marker_color='#ff7f0e'),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=time_labels, y=df_results['R_aFRR_Pos_MW'], name="+aFRR Committed (MW)",
               mode='lines+markers', line=dict(color='#2ca02c', width=2.5)),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=time_labels, y=df_results['R_aFRR_Neg_MW'], name="-aFRR Committed (MW)",
               mode='lines+markers', line=dict(color='#d62728', width=2.5)),
    row=2, col=1
)

# Chart 3: SOC & Cumulative Profit
hourly_profit = (
    (df_results['Spot_DA_EUR_MWh'] * df_results['P_discharge_MW'] - df_results['Spot_DA_EUR_MWh'] * df_results['P_charge_MW']) +
    (df_results['aFRR_Pos_EUR_MW'] * df_results['R_aFRR_Pos_MW']) +
    (df_results['aFRR_Neg_EUR_MW'] * df_results['R_aFRR_Neg_MW']) -
    (BESS_CONFIG['deg_cost'] * (df_results['P_charge_MW'] + df_results['P_discharge_MW']))
)
cumulative_profit = np.cumsum(hourly_profit)

fig.add_trace(
    go.Scatter(x=time_labels, y=df_results['SOC_Pct'], name="Battery SOC (%)",
               line=dict(color='#9467bd', width=3), fill='tozeroy', fillcolor='rgba(148, 103, 189, 0.15)'),
    row=3, col=1, secondary_y=False
)
fig.add_trace(
    go.Scatter(x=time_labels, y=cumulative_profit, name="Cumulative Net Profit (€)",
               line=dict(color='#00cc96', width=3)),
    row=3, col=1, secondary_y=True
)

# Layout Configuration: Moving legend to the right side and setting font color to black
fig.update_layout(
    height=1100,
    title_text="<b>Utility-Scale BESS Multi-Market Co-Optimization (EPEX Spot DA + Regelleistung aFRR)</b>",
    title_font_size=15,
    title_x=0.5,
    title_y=0.98,
    margin=dict(t=120, b=60, l=60, r=220),  # ایجاد فضای خالی در سمت راست برای قرارگیری لجند
    hovermode="x unified",
    legend=dict(
        orientation="v",         # چینش عمودی لجند
        yanchor="top",
        y=0.95,
        xanchor="left",
        x=1.02,                  # انتقال به بیرون از کادر نمودار در سمت راست
        font=dict(color="black", size=11), # رنگ متن کاملاً مشکی و خوانا
        bgcolor="rgba(255, 255, 255, 0.9)",
        bordercolor="rgba(0, 0, 0, 0.2)",
        borderwidth=1
    )
)

# Axis Titles
fig.update_yaxes(title_text="Spot Price (€/MWh)", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="aFRR Cap. (€/MW)", row=1, col=1, secondary_y=True)
fig.update_yaxes(title_text="Power/Reserve (MW)", row=2, col=1)
fig.update_yaxes(title_text="Battery SOC (%)", row=3, col=1, secondary_y=False)
fig.update_yaxes(title_text="Cum. Profit (€)", row=3, col=1, secondary_y=True)

fig.show()

In [12]:
# Cell 7: Export All Results, CSV, and Plots into a Downloadable ZIP Package

import os
import zipfile
import pandas as pd

# ۱. ایجاد پوشه ذخیره‌سازی خروجی‌ها
output_dir = "bess_output_results"
os.makedirs(output_dir, exist_ok=True)

# ۲. ذخیره دیتافریم نتایج به صورت فایل CSV
csv_filename = os.path.join(output_dir, "bess_co_optimization_results.csv")
df_results.to_csv(csv_filename, index=False)

# ۳. ذخیره نمودار Plotly به صورت فایل HTML (تعاملی) و PNG (تصویر ثابت)
html_filename = os.path.join(output_dir, "bess_market_dashboard.html")
fig.write_html(html_filename)

png_filename = os.path.join(output_dir, "bess_market_dashboard.png")
try:
    fig.write_image(png_filename, scale=2) # نیازمند kaleido در صورت خروجی عکس
    print("✅ تصویر داشبورد با موفقیت ذخیره شد.")
except Exception as e:
    print(f"⚠️ ذخیره عکس نیازمند کتابخانه kaleido است (فایل HTML و CSV ذخیره شدند). خطا: {e}")

# ۴. گزارش متنی خلاصه عملکرد (Summary Report)
report_text = f"""
==================================================
UTILITY-SCALE BESS CO-OPTIMIZATION REPORT (GERMANY)
==================================================
Optimization Status: Success
Net Daily Profit: {net_daily_profit:,.2f} EUR
--------------------------------------------------
- Spot Arbitrage Revenue: {spot_arbitrage_rev:,.2f} EUR
- aFRR Positive Capacity Revenue: {afrr_pos_rev:,.2f} EUR
- aFRR Negative Capacity Revenue: {afrr_neg_rev:,.2f} EUR
- Total Degradation Cost: -{degradation_cost:,.2f} EUR
==================================================
"""

report_filename = os.path.join(output_dir, "optimization_summary_report.txt")
with open(report_filename, "w", encoding="utf-8") as f:
    f.write(report_text)

# ۵. فشرده‌سازی تمام فایل‌ها در قالب یک فایل ZIP
zip_filename = "bess_results_bundle.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for foldername, subfolders, filenames in os.walk(output_dir):
        for filename in filenames:
            file_path = os.path.join(foldername, filename)
            zipf.write(file_path, arcname=os.path.basename(file_path))

print("\n" + "="*50)
print(report_text)
print("="*50)
print(f"📦 تمامی فایل‌ها با موفقیت فشرده و آماده دانلود شدند:")
print(f"👉 نام فایل فشرده در پوشه ژوپیتر: {zip_filename}")
print("="*50)

⚠️ ذخیره عکس نیازمند کتابخانه kaleido است (فایل HTML و CSV ذخیره شدند). خطا: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



UTILITY-SCALE BESS CO-OPTIMIZATION REPORT (GERMANY)
Optimization Status: Success
Net Daily Profit: 13,638.67 EUR
--------------------------------------------------
- Spot Arbitrage Revenue: 510.21 EUR
- aFRR Positive Capacity Revenue: 7,237.60 EUR
- aFRR Negative Capacity Revenue: 5,925.45 EUR
- Total Degradation Cost: -34.59 EUR

📦 تمامی فایل‌ها با موفقیت فشرده و آماده دانلود شدند:
👉 نام فایل فشرده در پوشه ژوپیتر: bess_results_bundle.zip
